# 第 61 课｜价值函数、Return 与 GAE

**从“完整回答得到一个奖励”，走向“每一步行动应当分到多少学习信号”。**

上一课研究概率比、裁剪和旧策略。本课补上 Actor-Critic 中的价值估计：先用一个**可以精确求解的有限环境**核对 Bellman 方程、TD 残差、折扣回报和 GAE，再用真实 PyTorch 网络更新 Actor 与 Critic。

**本课文件独立。** Notebook 内嵌全部代码，不依赖上一课源码；完整 `.py` 文件可单独运行。PyTorch＋标准库、默认CPU、离线、不下载模型。不是完整 PPO，也不是预训练 LLM 的效果评测。

首先要记住：**terminated（真实终止）与 truncated（外部时间上限）不同**。把它们混成 `done`，价值目标可能系统性偏低。

## 学习目标

完成后，你应能亲手验证：

1. 折扣 Return 与一时刻 TD target 的关系，以及如何使用真正的终止状态决定 bootstrap。
2. GAE 的反向递推，为什么 λ=0 是单步 TD，λ=1 与带正确尾部的折扣 Return 关联。
3. 截断时为什么仍需估计最后观察的价值，但不能把另一回合的优势接过来。
4. Actor 如何使用优势更新概率，Critic 如何回归固定的 λ-return，且梯度不会互相串线。
5. 怎样使用精确 Bellman 参考与固定评测，避免把训练 loss 当成真正的策略表现。

建议先完成第1–5节的数学实验，再运行小模型，最后做独立练习。长实现单元可折叠阅读。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第61课：价值函数、Return和GAE。

独立离线CPU PyTorch教学：精确Bellman参考、时间限制/截断处理、手算GAE、
分开更新actor与critic，并比较TD(0)、GAE(lambda=.95)、lambda-return。
不下载模型/语料、不调用Gymnasium、不训练预训练LLM、不实现完整PPO。
"""
from __future__ import annotations


import argparse
import copy
import hashlib
import json
import math
import platform
import random
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("本课需要PyTorch。请使用已有的PyTorch Python环境。") from exc

VERSION = "lesson61-gae-v1"
SOURCES = {
    "gae_paper": "https://arxiv.org/abs/1506.02438",
    "gym_time_limits": "https://gymnasium.farama.org/v1.1.1/tutorials/gymnasium_basics/handling_time_limits/",
    "spinningup_vpg": "https://spinningup.openai.com/en/latest/algorithms/vpg.html",
    "spinningup_ppo": "https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html",
    "torch_autograd": "https://docs.pytorch.org/docs/stable/notes/autograd.html",
}

class LessonError(ValueError):
    """本课程协议错误；不静默修改任务定义。"""


def require(condition, message):
    if not bool(condition):
        raise LessonError(message)


def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def sha256(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def tensor_hash(named):
    h = hashlib.sha256()
    for name, tensor in sorted(named.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()


def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)


@dataclass(frozen=True)
class Config:
    seed: int = 6101
    gamma: float = 0.95
    gae_lambda: float = 0.95
    rollout_horizon: int = 6
    updates: int = 100
    episodes_per_update: int = 24
    actor_lr: float = 0.035
    critic_lr: float = 0.02
    actor_hidden: int = 24
    critic_hidden: int = 24
    value_weight: float = 1.0
    entropy_coef: float = 0.01
    grad_clip: float = 1.0
    eval_episodes: int = 200

    def validate(self):
        for name in ("seed", "rollout_horizon", "updates", "episodes_per_update", "actor_hidden",
                     "critic_hidden", "eval_episodes"):
            value = getattr(self, name)
            require(type(value) is int and value >= (0 if name == "seed" else 1), name + "无效")
        require(self.rollout_horizon >= 3 and self.rollout_horizon <= 16, "rollout_horizon不在3..16")
        for name in ("gamma", "gae_lambda", "actor_lr", "critic_lr", "value_weight", "entropy_coef", "grad_clip"):
            val = getattr(self, name)
            require(type(val) in (float, int) and math.isfinite(val), name + "必须为有限数值")
        require(0 < self.gamma <= 1 and 0 <= self.gae_lambda <= 1, "gamma/lambda不在合法范围")
        require(self.actor_lr > 0 and self.critic_lr > 0 and self.value_weight > 0 and
                self.entropy_coef >= 0 and self.grad_clip > 0, "学习率/正则/裁剪非法")

In [2]:
torch.set_num_threads(1)
seed_all(6101)
OUTPUT_ROOT = 'lesson_61_outputs'
RUN_FULL_TRAINING = True
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('本课执行设备：CPU；不调用GPU，也不下载模型。')

Python: 3.13.5
PyTorch: 2.10.0+cpu
本课执行设备：CPU；不调用GPU，也不下载模型。


## 1. 有一个目标位置的继续任务

固定环境中，非终止位置是 `0、1、2`，真正目标为 `3`。动作只有两种：

| 动作 | 环境变化 |
|---|---|
| 0，等待 | 留在当前位置 |
| 1，前进 | 前进一格；达到3则真终止 |

每步成本 −0.01；进入目标额外奖励+1，因此最后一步得到+0.99。

环境的**底层过程可以无限继续**（未到目标时），但为了便于采样，外部包装器规定最多6步。第6步没有成功则标记 `truncated=True`。这不是任务本身的真实终止。

如果你的任务在定义中就有有限时域终点，那么该终点属于 **terminated**，而且为了保持马尔可夫性，通常需要把剩余时间包含在状态中。我们的时间上限明确在底层任务之外。[Gymnasium：Handling Time Limits](https://gymnasium.farama.org/v1.1.1/tutorials/gymnasium_basics/handling_time_limits/)

**不把“先运行失败”“网络断开”“模型生成中断”自动当成奖励0或MDP真终止。**

In [3]:
GOAL = 3  # 状态0、1、2；达到3是真终止
ACTION_WAIT, ACTION_ADVANCE = 0, 1
STEP_COST, SUCCESS_BONUS = -0.01, 1.0

@dataclass(frozen=True)
class Step:
    obs: int
    action: int
    reward: float
    next_obs: int
    terminated: bool
    truncated: bool
    old_logp: float


def validate_step(step):
    require(isinstance(step, Step), "Step类型错误")
    require(type(step.obs) is int and 0 <= step.obs < GOAL, "当前obs必须是非终止位置")
    require(type(step.next_obs) is int and 0 <= step.next_obs <= GOAL, "next_obs无效")
    require(type(step.action) is int and step.action in (ACTION_WAIT, ACTION_ADVANCE), "动作非法")
    require(type(step.terminated) is bool and type(step.truncated) is bool,
            "terminated/truncated必须是bool")
    require(not (step.terminated and step.truncated), "真实终止与外部截断在本课程中互斥")
    require(step.next_obs == min(step.obs + step.action, GOAL), "环境动态不匹配")
    require(step.terminated == (step.next_obs == GOAL), "终止状态不匹配")
    expected_reward = STEP_COST + (SUCCESS_BONUS if step.terminated else 0)
    require(math.isfinite(step.reward) and abs(step.reward - expected_reward) < 1e-10,
            "奖励必须来自固定环境，而不是模型自报")
    require(type(step.old_logp) is float and math.isfinite(step.old_logp) and step.old_logp <= 1e-6,
            "需要有效的行为log概率")


def validate_episode(episode, horizon):
    require(type(episode) is list and 1 <= len(episode) <= horizon, "episode长度错误")
    require(episode[0].obs == 0, "每回合必须从位置0开始")
    for i, step in enumerate(episode):
        validate_step(step)
        if i < len(episode) - 1:
            require(not step.terminated and not step.truncated, "回合结束后仍有步骤")
            require(episode[i + 1].obs == step.next_obs, "下一步观察不连续")
    require(episode[-1].terminated or episode[-1].truncated, "回合尾部必须有明确终止状态")
    require(not episode[-1].truncated or len(episode) == horizon,
            "外部时间上限只能在约定horizon触发")


def transition(pos, action, t, horizon):
    require(type(pos) is int and 0 <= pos < GOAL, "pos非法")
    require(type(action) is int and action in (0, 1), "action非法")
    require(type(t) is int and 0 <= t < horizon, "时间步非法")
    nxt = min(pos + action, GOAL)
    terminated = nxt == GOAL
    truncated = t + 1 >= horizon and not terminated
    reward = STEP_COST + (SUCCESS_BONUS if terminated else 0.)
    return nxt, reward, terminated, truncated

In [4]:
for t,pos,action in [(0,0,0),(1,1,1),(5,2,0),(5,2,1)]:
    print({'time':t,'state':pos,'action':action,'outcome':transition(pos,action,t,6)})

{'time': 0, 'state': 0, 'action': 0, 'outcome': (0, -0.01, False, False)}
{'time': 1, 'state': 1, 'action': 1, 'outcome': (2, -0.01, False, False)}
{'time': 5, 'state': 2, 'action': 0, 'outcome': (2, -0.01, False, True)}
{'time': 5, 'state': 2, 'action': 1, 'outcome': (3, 0.99, True, False)}


## 2. Return：未来奖励要按 γ 折扣

令 $r_t$ 为本步奖励，折扣回报满足：

\[G_t=r_t+\gamma G_{t+1}\]

例如两步奖励为 `[1, 2]`，γ=0.5：第二步的回报是2，第一步为 $1+0.5\times2=2$。

如果只采到一段非终止片段，末尾还有尾部价值 $V(s_T)$，就用 $G_T=V(s_T)$ 再倒推。这样得到的是**带 bootstrap 的回报估计**，不是把未观察到的未来强行当成零。

In [5]:
def checked_numeric(values, name):
    require(isinstance(values, (list, tuple)) and len(values) > 0, name + "必须为非空序列")
    require(all(type(x) in (int, float) and math.isfinite(x) for x in values), name + "必须有限")


def discounted_returns(rewards, gamma, bootstrap=0.0):
    """时间倒序累计，最后一个非终止cut使用V(final_obs)作为尾部。"""
    checked_numeric(rewards, "rewards")
    require(type(gamma) in (int, float) and 0 < gamma <= 1, "gamma非法")
    require(type(bootstrap) in (int, float) and math.isfinite(bootstrap), "bootstrap非法")
    acc = float(bootstrap)
    output = [0.] * len(rewards)
    for i in range(len(rewards)-1, -1, -1):
        acc = float(rewards[i]) + gamma*acc
        output[i] = acc
    return output


def generalized_advantage_estimation(episodes, values, next_values, gamma, lam):
    """独立episode：delta=r+gamma*(1-terminated)V(next)-V(s)。

    截断时delta保留V(final_obs)，但由于下一条记录是新回合，trace递推停止。
    自然rollout数组末尾也停止trace，若未终止则必须保留下一状态bootstrap。
    返回 A 与 lambda-return=A+V_old，不连接critic梯度。
    """
    require(type(episodes) is list and bool(episodes), "需要回合列表")
    require(type(gamma) in (float, int) and 0 < gamma <= 1, "gamma非法")
    require(type(lam) in (float, int) and 0 <= lam <= 1, "lambda非法")
    count = sum(len(ep) for ep in episodes)
    checked_numeric(values, "values")
    checked_numeric(next_values, "next_values")
    require(len(values) == len(next_values) == count, "值函数数组长度与步骤数量不符")
    advantages, targets, deltas = [], [], []
    offset = 0
    for ep in episodes:
        require(len(ep) > 0, "空回合")
        for j, step in enumerate(ep):
            validate_step(step)
            if j < len(ep)-1:
                require(not (step.terminated or step.truncated), "episode结束后仍有下一条步骤")
                require(ep[j+1].obs == step.next_obs, "GAE回合内部不连续")
        local_values = values[offset:offset + len(ep)]
        local_next = next_values[offset:offset + len(ep)]
        offset += len(ep)
        local_d = [step.reward + gamma*(0. if step.terminated else nv) - v
                   for step,v,nv in zip(ep,local_values,local_next)]
        local_adv = [0.] * len(ep)
        ahead = 0.
        for j in range(len(ep)-1, -1, -1):
            # 回合结束或采样切断后，不能把下一回合的优势连接过来。
            boundary = ep[j].terminated or ep[j].truncated
            ahead = local_d[j] + gamma*lam*(0. if boundary else ahead)
            local_adv[j] = ahead
        deltas.extend(local_d)
        advantages.extend(local_adv)
        targets.extend([a+v for a,v in zip(local_adv,local_values)])
    return {"delta":deltas, "advantage":advantages, "lambda_target":targets}


def exact_policy_values(forward_probabilities, gamma=0.95):
    """Bellman: 在底层无限时域、位置马尔可夫过程里计算精确V^pi。

    外部6步截断不属于底层MDP，所以不能把它写进该值函数。
    """
    checked_numeric(forward_probabilities, "forward_probabilities")
    require(len(forward_probabilities) == GOAL, "每个非终止位置一个前进概率")
    require(all(0 <= v <= 1 for v in forward_probabilities), "动作概率不在0..1")
    require(type(gamma) in (int,float) and 0 < gamma < 1, "无穷时域要求gamma小于1")
    values = [0.] * GOAL
    for s in range(GOAL-1, -1, -1):
        p = float(forward_probabilities[s])
        forward_value = SUCCESS_BONUS + STEP_COST if s == GOAL-1 else STEP_COST + gamma*values[s+1]
        values[s] = (p*forward_value + (1-p)*STEP_COST)/(1 - gamma*(1-p))
    return values


def finite_horizon_success_probability(forward_probabilities, horizon):
    """固定外部horizon内到达目标的精确概率，不等于底层无限时域V。"""
    require(type(horizon) is int and horizon >= 1, "horizon非法")
    checked_numeric(forward_probabilities, "policy")
    require(len(forward_probabilities)==GOAL and all(0 <= x <= 1 for x in forward_probabilities),
            "策略概率非法")
    active = [1., 0., 0.]
    finished = 0.
    for _ in range(horizon):
        new = [0.] * GOAL
        for s,pop in enumerate(active):
            prob = forward_probabilities[s]
            new[s] += pop*(1-prob)
            if s == GOAL-1:
                finished += pop*prob
            else:
                new[s+1] += pop*prob
        active = new
    require(abs(finished+sum(active)-1)<1e-9, "总概率不归一")
    return finished


def gae_examples():
    gamma, lam = .95, .8
    ep=[Step(1,1,-.01,2,False,False,-.7),Step(2,0,-.01,2,False,True,-.7)]
    values=[.6,.7]
    next_values=[.7,.8]
    result=generalized_advantage_estimation([ep],values,next_values,gamma,lam)
    # 若误把截断当终止，会从最后一步少算gamma*0.8=0.76，并传到上一步。
    wrong_last=-.01-.7
    correct_last=-.01+gamma*.8-.7
    lam_0=generalized_advantage_estimation([ep],values,next_values,gamma,0)
    lam_1=generalized_advantage_estimation([ep],values,next_values,gamma,1)
    return {"gamma":gamma,"lambda":lam,"values":values,"next_values":next_values,
            "gae":result,"wrong_truncation_delta":wrong_last,
            "correct_truncation_delta":correct_last,
            "lost_bootstrap":gamma*.8,
            "lambda_0":lam_0,"lambda_1":lam_1,
            "mc_with_bootstrap":discounted_returns([s.reward for s in ep],gamma,bootstrap=.8),
            "wrong_no_bootstrap_return":discounted_returns([s.reward for s in ep],gamma,bootstrap=0)}




def exact_time_limit_bias(forward_probs=(.5,.5,.5),gamma=.95,horizon=6):
    """精确枚举2^horizon个独立动作。只算位置0起点的截断自举偏差。"""
    require(type(horizon) is int and 1<=horizon<=12, "枚举horizon过大")
    true_v=exact_policy_values(list(forward_probs),gamma)
    require(all(0 < p < 1 for p in forward_probs), "枚举的每个动作必须有非零概率")
    expected_good=0.; expected_wrong=0.; covered_probability=0.; truncated_probability=0.
    samples=[]
    for bits in __import__('itertools').product((0,1),repeat=horizon):
        pos=0; probability=1.;ep=[]
        for t,action in enumerate(bits):
            probability*=forward_probs[pos] if action else 1-forward_probs[pos]
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,math.log(forward_probs[pos] if action else 1-forward_probs[pos])))
            if terminated or truncated:break
            pos=nxt
        # 对在终点前提前终止的序列，只计一次动作前缀，不能按完整bits重复累加。
        if ep[-1].terminated and bits[len(ep):] != (0,)*(horizon-len(ep)):
            continue
        # 退出前缀的概率已包含全部已执行动作；未使用位不参与概率。
        tail=0. if ep[-1].terminated else true_v[ep[-1].next_obs]
        good=discounted_returns([e.reward for e in ep],gamma,tail)[0]
        wrong=discounted_returns([e.reward for e in ep],gamma,0.)[0]
        covered_probability+=probability
        truncated_probability+=probability*ep[-1].truncated
        expected_good+=probability*good
        expected_wrong+=probability*wrong
        values=[true_v[e.obs] for e in ep]
        next_values=[0. if e.terminated else true_v[e.next_obs] for e in ep]
        samples.append((probability,ep,values,next_values))
    require(abs(covered_probability-1)<1e-9,"枚举覆盖概率不足")
    var_by_lambda={}
    for lam in (0.,.5,.95,1.):
        means=0.;second=0.
        for probability,ep,values,next_values in samples:
            a=generalized_advantage_estimation([ep],values,next_values,gamma,lam)['advantage'][0]
            means+=probability*a;second+=probability*a*a
        var_by_lambda[str(lam)]={"mean_advantage_at_start":means,
                                  "variance_at_start":max(0.,second-means*means)}
    return {"true_continuing_value_start":true_v[0],"truncation_probability":truncated_probability,
            "proper_bootstrap_expected_return":expected_good,
            "wrong_treat_time_limit_as_terminal_return":expected_wrong,
            "lost_expected_value":expected_good-expected_wrong,
            "lambda_gradient_noise_example":var_by_lambda,
            "scope":"exact_enumeration_fixed_policy_time_limit_outside_underlying_MDP"}

In [6]:
print('无尾部：',discounted_returns([1,2],.5))
print('尾部V=3：',discounted_returns([1,2],.5,bootstrap=3.0))
print('一步正确时间限制：',gae_examples()['correct_truncation_delta'])
print('一步误当终止：',gae_examples()['wrong_truncation_delta'])

无尾部： [2.0, 2.0]
尾部V=3： [2.75, 3.5]
一步正确时间限制： 0.050000000000000044
一步误当终止： -0.71


## 3. 价值函数与 Bellman 等式

$V^{\pi}(s)$ 表示从状态 $s$ 按策略 $\pi$ 继续运行的**预期折扣奖励和**，而不是“这一动作有多好”，也不是“模型的置信分”。

对本课的继续任务：

\[V^\pi(s)=\sum_a\pi(a|s)\big(r(s,a)+\gamma V^\pi(s')\big)\]

真正进入目标的转移，后续价值是零。本环境状态有限、转移已知，所以可以直接求解 Bellman 方程，得到策略价值的**精确参考**。

现实的大语言模型没有这样的完整有限转移表，因此不能把本课能精确求解的能力直接搬过去。

In [7]:
uniform_p = [0.5,0.5,0.5]
exact_values = exact_policy_values(uniform_p,gamma=.95)
finite_success = finite_horizon_success_probability(uniform_p,horizon=6)
print('p(前进)=0.5时底层继续价值：',exact_values)
print('6步内到达目标的精确概率：',finite_success)
print('全等待的继续价值：',exact_policy_values([0.,0.,0.],.95))

p(前进)=0.5时底层继续价值： [0.7277399848828416, 0.8253968253968251, 0.9333333333333332]
6步内到达目标的精确概率： 0.65625
全等待的继续价值： [-0.19999999999999982, -0.19999999999999982, -0.19999999999999982]


## 4. 一步 TD residual：真终止才把尾部设为零

本课使用：

\[\delta_t=r_t+\gamma(1-\mathrm{terminated}_t)V(s_{t+1})-V(s_t)\]

`truncated=True` 时，最后观察仍是底层MDP的一个合法状态。它需要 bootstrap；但**回合的数据已经断开**，下一回合从0重启，不能把重启后的优势串进当前回合。

例如设当前值为0.7、真实下一状态值为0.8，奖励−0.01、γ=0.95：

\[\delta=-0.01+0.95\times0.8-0.7=0.05\]

误把时间截断当真终止，则得到−0.71；少掉 **0.76**。这不是四舍五入误差。

In [8]:
demo=gae_examples()
print('正确TD残差：',demo['correct_truncation_delta'])
print('错误TD残差：',demo['wrong_truncation_delta'])
print('丢失的bootstrap：',demo['lost_bootstrap'])
print('截断后的最后一步标签:',demo['gae'])

正确TD残差： 0.050000000000000044
错误TD残差： -0.71
丢失的bootstrap： 0.76
截断后的最后一步标签: {'delta': [0.05499999999999994, 0.050000000000000044], 'advantage': [0.09299999999999997, 0.050000000000000044], 'lambda_target': [0.693, 0.75]}


## 5. GAE：从单步残差向后递推

对每步残差 $\delta_t$，定义：

\[\widehat A_t=\delta_t+\gamma\lambda\widehat A_{t+1}\]

在真终止、外部截断和片段边界处，**优势递推停止**。但边界的本步 $\delta_t$ 是否 bootstrap，仍由**真终止标志**决定。

| 取值 | 解释 |
|---|---|
| $\lambda=0$ | 单步 TD 残差 |
| $0<\lambda<1$ | 用指数衰减综合未来 TD 残差 |
| $\lambda=1$ | 沿该片段累计折扣残差；配合正确尾部时，$\widehat A+V$ 等于带 bootstrap 的回报 |

Critic 回归的标签是：

\[\widehat R_t^{\lambda}=\widehat A_t+V_{\rm old}(s_t)\]

**形成训练标签时必须固定旧值。** 如果直接让 Critic 目标在反向过程中随自己的预测变化，会优化另一个计算图。

原始 GAE 工作讨论了估计偏差与方差的权衡，而不是宣称某个 λ 对所有任务最优。[GAE论文](https://arxiv.org/abs/1506.02438)

In [9]:
for label in ['lambda_0','gae','lambda_1']:
    item=demo[label]
    print(label,'优势=',item['advantage'],'lambda目标=',item['lambda_target'])
print('λ=1时带尾部Return:',demo['mc_with_bootstrap'])

lambda_0 优势= [0.05499999999999994, 0.050000000000000044] lambda目标= [0.6549999999999999, 0.75]
gae 优势= [0.09299999999999997, 0.050000000000000044] lambda目标= [0.693, 0.75]
lambda_1 优势= [0.10249999999999998, 0.050000000000000044] lambda目标= [0.7024999999999999, 0.75]
λ=1时带尾部Return: [0.7024999999999999, 0.75]


## 6. 精确枚举：外部截断会造成多大偏差？

这次不靠随机抽样猜测。固定三个位置的前进概率均为0.5，列举最多6步的所有动作前缀，按真实概率加权。

我们比较：

1. 对外部截断仍使用真正的下一状态价值。
2. 错误地在外部截断处把所有未来价值设为0。

第一种期望应与精确 Bellman 继续价值一致。第二种因为把可能的后续成功抹掉，会偏低。

同时枚举 λ=0、0.5、0.95、1 时起点优势的方差。**这里使用精确真实的 Critic**；换成学习中的近似 Critic，偏差/方差关系可能不同。

In [10]:
bias = exact_time_limit_bias()
print('精确继续价值：',bias['true_continuing_value_start'])
print('正确bootstrap的期望：',bias['proper_bootstrap_expected_return'])
print('错误截断处理的期望：',bias['wrong_treat_time_limit_as_terminal_return'])
print('系统性少算：',bias['lost_expected_value'])
print('不同lambda在本例的优势方差：')
for key,item in bias['lambda_gradient_noise_example'].items():
    print(key,item)

精确继续价值： 0.7277399848828416
正确bootstrap的期望： 0.727739984882842
错误截断处理的期望： 0.501697839453125
系统性少算： 0.226042145429717
不同lambda在本例的优势方差：
0.0 {'mean_advantage_at_start': 0.0, 'variance_at_start': 0.002151753698876044}
0.5 {'mean_advantage_at_start': 0.0, 'variance_at_start': 0.0028604751243776205}
0.95 {'mean_advantage_at_start': 6.938893903907228e-18, 'variance_at_start': 0.008598104555737713}
1.0 {'mean_advantage_at_start': 6.938893903907228e-18, 'variance_at_start': 0.010489943749847463}


## 7. Actor 与 Critic：两条独立的梯度路径

本课真正建立两个 PyTorch 网络：

| 网络 | 输入与输出 | 更新目标 |
|---|---|---|
| Actor | 位置索引 → 等待/前进两个 logits | 增大正优势动作的 log 概率、减小负优势动作的概率 |
| Critic | 位置索引 → 单个价值估计 | 回归固定的 $\lambda$-return |

Actor 训练损失：

\[L_{actor}=-\mathrm{mean}_t(\log\pi_\theta(a_t|s_t)\widehat A_t)-\eta\,\mathrm{mean}_tH(\pi_\theta(\cdot|s_t))\]

Critic 训练损失：

\[L_{critic}=\tfrac12\mathrm{mean}_t(V_\phi(s_t)-\widehat R_t^{\lambda})^2\]

Actor与Critic参数分开、优化器分开。优势和旧返回标签不持有 Critic 的计算图。本课每批 on-policy 采样只更新**一次**；没有 PPO-Clip、重复旧批次、价值裁剪或 GAE 与价值网络的联合大模型架构。

由于环境很小，我们还能逐状态比较 Critic 预测与同一当前策略的**精确继续价值**。

In [11]:
class Actor(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,2))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "actor状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "actor状态越界")
        return self.net(states)


class Critic(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,1))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "critic状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "critic状态越界")
        return self.net(states).squeeze(-1)


def actor_probabilities(actor):
    with torch.no_grad():
        actor.eval()
        probs=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)[:,ACTION_ADVANCE]
    return probs.tolist()


def value_prediction(critic):
    with torch.no_grad():
        critic.eval()
        return critic(torch.arange(GOAL,dtype=torch.long)).tolist()


def value_error(critic,actor,gamma):
    reference=exact_policy_values(actor_probabilities(actor),gamma)
    prediction=value_prediction(critic)
    return {"predicted":prediction,"exact_for_current_policy":reference,
            "rmse":math.sqrt(sum((a-b)**2 for a,b in zip(prediction,reference))/GOAL),
            "scope":"exact_infinite_horizon_value_of_current_policy_not_finite_time_limit_return"}

In [12]:
seed_all(6111)
a=Actor(16);v=Critic(16)
print('Actor参数：',sum(p.numel() for p in a.parameters()))
print('Critic参数：',sum(p.numel() for p in v.parameters()))
print('两侧参数ID互不重叠：',not {id(p) for p in a.parameters()} & {id(p) for p in v.parameters()})
print('前进概率：',actor_probabilities(a))
print('Critic初始预测与Bellman参考：',value_error(v,a,.95))

Actor参数： 354
Critic参数： 337
两侧参数ID互不重叠： True
前进概率： [0.43758708238601685, 0.4660356640815735, 0.5406534671783447]
Critic初始预测与Bellman参考： {'predicted': [-0.06740206480026245, -0.3726540803909302, 0.6728180646896362], 'exact_for_current_policy': [0.7155539326603839, 0.8256738402788648, 0.9415079659895499], 'rmse': 0.8408733877100538, 'scope': 'exact_infinite_horizon_value_of_current_policy_not_finite_time_limit_return'}


## 8. 收集 on-policy 回合，锁定优势与返回目标

每回合从位置0开始，按当前 Actor 概率采样，直到真实终止或最多6步截断。奖励由环境返回，不接受模型自报。记录旧 log 概率，在更新前对照当前 Actor，防止使用参数已经改变的旧数据。

构建目标时，Critic 在 `no_grad()` 下提供 $V(s)$ 和 $V(s')$；真终止使用0，截断保留真正的最后状态值。`advantages` 与 `lambda_returns` 转成固定数值，Actor/Critic 分别使用。

**真实算法中的批次可能包含并行环境和跨 rollout 的片段。** 本课用独立完整回合简化边界，每个回合验证状态连续性与终止/截断标志。

In [13]:
def collect_rollouts(actor, episodes, horizon, rng):
    require(type(episodes) is int and episodes >= 1, "episodes非法")
    require(type(horizon) is int and horizon >= 3, "horizon非法")
    result=[]
    actor.eval()
    with torch.no_grad():
        p=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)
    for _ in range(episodes):
        pos=0;ep=[]
        for t in range(horizon):
            action=1 if rng.random()<float(p[pos,1]) else 0
            logp=math.log(float(p[pos,action]))
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,logp))
            if terminated or truncated:
                break
            pos=nxt
        validate_episode(ep,horizon)
        result.append(ep)
    return result


def compute_targets(episodes,critic,gamma,lam,bootstrap_truncation=True):
    require(type(bootstrap_truncation) is bool, "bootstrap标志必须为bool")
    states=[s.obs for ep in episodes for s in ep]
    nextstates=[s.next_obs if not s.terminated else 0 for ep in episodes for s in ep]
    with torch.no_grad():
        values=critic(torch.tensor(states,dtype=torch.long)).tolist()
        nextvalues=critic(torch.tensor(nextstates,dtype=torch.long)).tolist()
    for i,step in enumerate(s for ep in episodes for s in ep):
        if step.terminated or (step.truncated and not bootstrap_truncation):
            nextvalues[i]=0.
    target=generalized_advantage_estimation(episodes,values,nextvalues,gamma,lam)
    return {"states":states,"next_states":nextstates,"values_old":values,
            "next_values_old":nextvalues,"advantages":target['advantage'],
            "lambda_returns":target['lambda_target'],"deltas":target['delta'],
            "old_logps":[s.old_logp for ep in episodes for s in ep],
            "actions":[s.action for ep in episodes for s in ep],
            "rewards":[s.reward for ep in episodes for s in ep],
            "terminated":[s.terminated for ep in episodes for s in ep],
            "truncated":[s.truncated for ep in episodes for s in ep],
            "episodes":len(episodes),"steps":len(states),
            "bootstrap_truncation":bootstrap_truncation,
            "scope":"on_policy_fixed_targets_detached_from_critic"}


def optimize_actor_critic(actor,critic,targets,optim_actor,optim_critic,config):
    require(len(targets['states'])>0,"空训练数据")
    require(set(id(p) for g in optim_actor.param_groups for p in g['params'])==
            {id(p) for p in actor.parameters()},"actor优化器对象错配")
    require(set(id(p) for g in optim_critic.param_groups for p in g['params'])==
            {id(p) for p in critic.parameters()},"critic优化器对象错配")
    states=torch.tensor(targets['states'],dtype=torch.long)
    actions=torch.tensor(targets['actions'],dtype=torch.long)
    old_lp=torch.tensor(targets['old_logps'],dtype=torch.float32)
    advantages=torch.tensor(targets['advantages'],dtype=torch.float32)
    fixed_returns=torch.tensor(targets['lambda_returns'],dtype=torch.float32)
    require(not advantages.requires_grad and not fixed_returns.requires_grad,"优势与回报目标必须固定")
    actor.train();critic.train()
    logits=actor(states)
    logprobs=F.log_softmax(logits,dim=-1)
    selected=logprobs.gather(-1,actions[:,None]).squeeze(-1)
    require(torch.allclose(selected.detach(),old_lp,rtol=1e-4,atol=2e-5),
            "训练前策略已变化，样本不再属于当前策略")
    entropy=-(logprobs.exp()*logprobs).sum(-1)
    actor_loss=-(selected*advantages).mean()-config.entropy_coef*entropy.mean()
    predicted=critic(states)
    value_loss=.5*config.value_weight*(predicted-fixed_returns).square().mean()
    optim_actor.zero_grad(set_to_none=True);optim_critic.zero_grad(set_to_none=True)
    actor_loss.backward()
    require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in actor.parameters()),
            "actor梯度异常")
    require(all(p.grad is None for p in critic.parameters()),"actor backward 不应直接更新critic梯度")
    actor_g=nn.utils.clip_grad_norm_(actor.parameters(),config.grad_clip,error_if_nonfinite=True)
    value_loss.backward()
    require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in critic.parameters()),
            "critic梯度异常")
    critic_g=nn.utils.clip_grad_norm_(critic.parameters(),config.grad_clip,error_if_nonfinite=True)
    optim_actor.step();optim_critic.step()
    return {"actor_loss":float(actor_loss.detach()),"value_loss":float(value_loss.detach()),
            "mean_advantage":float(advantages.mean()),"advantage_std":float(advantages.std(unbiased=False)),
            "mean_entropy":float(entropy.mean().detach()),"actor_grad_norm":float(actor_g),
            "critic_grad_norm":float(critic_g),"training_steps":len(states)}

In [14]:
collected=collect_rollouts(a,episodes=5,horizon=6,rng=random.Random(6101))
print('回合长度：',[len(ep) for ep in collected])
print('回合末状态：',[(ep[-1].terminated,ep[-1].truncated) for ep in collected])
prepared=compute_targets(collected,v,gamma=.95,lam=.95)
print('采样步数：',prepared['steps'])
print('前五条TD残差：',[round(x,5) for x in prepared['deltas'][:5]])
print('前五条λ-return：',[round(x,5) for x in prepared['lambda_returns'][:5]])

回合长度： [3, 6, 6, 6, 5]
回合末状态： [(True, False), (False, True), (False, True), (True, False), (True, False)]
采样步数： 26
前五条TD残差： [-0.29662, 1.00183, 0.31718, -0.00663, -0.00663]
前五条λ-return： [0.79848, 0.91543, 0.99, -0.2819, -0.29773]


## 9. 真正执行一次更新，并核对参数身份

这个单元只使用刚刚收集的一批数据，没有根据训练结果反复补采成功样本。

Actor/Critic 各自建立优化器，先断言旧 log 概率与当前网络一致，再执行：清零、各自反向、各自梯度裁剪、各自更新。第二次直接复用同一个 prepared，在策略已改变后应被拒绝。

这一步能检查梯度链路正确，但不保证“每一次梯度更新都提高表现”。

In [15]:
opt_actor=torch.optim.Adam(a.parameters(),lr=.035)
opt_critic=torch.optim.Adam(v.parameters(),lr=.02)
a_hash=tensor_hash(a.state_dict());v_hash=tensor_hash(v.state_dict())
one_step=optimize_actor_critic(a,v,prepared,opt_actor,opt_critic,Config(actor_hidden=16,critic_hidden=16))
print('单步损失与梯度：',one_step)
print('Actor参数改变：',tensor_hash(a.state_dict())!=a_hash)
print('Critic参数改变：',tensor_hash(v.state_dict())!=v_hash)
try:
    optimize_actor_critic(a,v,prepared,opt_actor,opt_critic,Config(actor_hidden=16,critic_hidden=16))
except LessonError as error:
    print('旧数据被拒绝：',error)

单步损失与梯度： {'actor_loss': 0.32997238636016846, 'value_loss': 0.23192602396011353, 'mean_advantage': 0.47079548239707947, 'advantage_std': 0.49214187264442444, 'mean_entropy': 0.6874716281890869, 'actor_grad_norm': 0.28581011295318604, 'critic_grad_norm': 0.9871430397033691, 'training_steps': 26}
Actor参数改变： True
Critic参数改变： True
旧数据被拒绝： 训练前策略已变化，样本不再属于当前策略


## 10. 全量训练：三种 λ 从同一起点出发

接下来的实验使用相同初始网络参数、相同预生成回合采样种子表与固定训练配置；但模型更新以后，同一随机数可能产生不同动作，**真实采样回合不会完全相同**。

三个分支：`td0`（λ=0）、`gae`（λ=0.95）、`lambda1`（λ=1）。默认每条100次更新，每批24个回合，都是短小的CPU实验。

策略表现用两种方式评测：

1. 根据 Actor 的逐状态前进概率，**精确计算**六步内到达目标的概率。
2. 使用固定种子的200条回合，记录实际成功数量（存在采样噪声）。

Critic 另外与当前策略的底层继续 Bellman 价值比较，报告 RMSE。**不能把 Critic MSE、Actor loss 或“到达目标概率”叫成语言模型正确率。**

In [16]:
def evaluate_policy(actor,critic,gamma,horizon,rollouts=200,seed=2026):
    prob=actor_probabilities(actor)
    exact_continuing=exact_policy_values(prob,gamma)
    exact_success=finite_horizon_success_probability(prob,horizon)
    sampled=collect_rollouts(actor,rollouts,horizon,random.Random(seed))
    success=sum(ep[-1].terminated for ep in sampled)
    truncated=sum(ep[-1].truncated for ep in sampled)
    return {"forward_probabilities":prob,"exact_continuing_value":exact_continuing,
            "exact_success_within_limit":exact_success,
            "sampled_successes":success,"sampled_episodes":rollouts,
            "sampled_success_rate":success/rollouts,"sampled_truncations":truncated,
            "value_rmse":value_error(critic,actor,gamma)['rmse'],
            "note":"固定环境的精确计算＋有限采样；并非模型数学推理能力"}


def train_one_arm(cfg,lam,train_schedule,init_actor,init_critic):
    actor=copy.deepcopy(init_actor);critic=copy.deepcopy(init_critic)
    actor_opt=torch.optim.Adam(actor.parameters(),lr=cfg.actor_lr)
    critic_opt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    history=[]
    for update,episodes_seed in enumerate(train_schedule,1):
        eps=collect_rollouts(actor,cfg.episodes_per_update,cfg.rollout_horizon,
                             random.Random(episodes_seed))
        target=compute_targets(eps,critic,cfg.gamma,lam,True)
        stats=optimize_actor_critic(actor,critic,target,actor_opt,critic_opt,cfg)
        if update in (1,cfg.updates) or update%20==0:
            ev=evaluate_policy(actor,critic,cfg.gamma,cfg.rollout_horizon,40,seed=9990)
            history.append({"update":update,"losses":stats,
                            "exact_success_within_limit":ev['exact_success_within_limit'],
                            "value_rmse":ev['value_rmse'],
                            "terminations_in_train":sum(e[-1].terminated for e in eps),
                            "truncations_in_train":sum(e[-1].truncated for e in eps)})
    return actor,critic,history


def save_json(path,value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2,
                                      allow_nan=False)+"\n",encoding="utf-8")


def experiment(cfg=None,root='lesson_61_outputs',mode='all'):
    cfg=cfg or Config();cfg.validate()
    require(mode in ('all','td0','gae','lambda1'),"训练模式不支持")
    torch.set_num_threads(1)
    seed_all(cfg.seed)
    init_actor=Actor(cfg.actor_hidden)
    init_critic=Critic(cfg.critic_hidden)
    actor_hash=tensor_hash(init_actor.state_dict());critic_hash=tensor_hash(init_critic.state_dict())
    schedule_rng=random.Random(cfg.seed+1)
    # 相同batch采样种子与初始参数；后面分支策略不同，实际轨迹也不同。
    schedule=[schedule_rng.randrange(2**31) for _ in range(cfg.updates)]
    arms={'td0':0.0,'gae':cfg.gae_lambda,'lambda1':1.0}
    if mode!='all': arms={mode:arms[mode]}
    output_root=Path(root);output_root.mkdir(parents=True,exist_ok=True)
    run=output_root/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    run.mkdir(exist_ok=False)
    manifest={"version":VERSION,"status":"running","python":platform.python_version(),
              "torch":torch.__version__,"device":"cpu","config":asdict(cfg),"mode":mode,
              "sources":SOURCES,"boundaries":["synthetic_continuing_mdp_with_external_time_limit",
                "simple_MLP_actor_and_critic_not_llm","on_policy_one_update_per_batch",
                "not_full_PPO_no_GAE_critic_PPO_value_clipping","single_seed_no_algorithm_ranking"]}
    save_json(run/'manifest.json',manifest)
    try:
        base=evaluate_policy(init_actor,init_critic,cfg.gamma,cfg.rollout_horizon,cfg.eval_episodes)
        report={"reference_initial":base,"starting_actor_hash":actor_hash,
                "starting_critic_hash":critic_hash,"schedule_hash":sha256(schedule),
                "arms":{},"selection":"fixed_last_update_without_test_based_selection"}
        for name,lam in arms.items():
            print('训练分支',name,'lambda=',lam,flush=True)
            actor,critic,history=train_one_arm(cfg,lam,schedule,init_actor,init_critic)
            final=evaluate_policy(actor,critic,cfg.gamma,cfg.rollout_horizon,cfg.eval_episodes)
            report['arms'][name]={"lambda":lam,"history":history,"final":final,
                    "total_train_episodes":cfg.updates*cfg.episodes_per_update,
                    "initial_actor_hash_unchanged":tensor_hash(init_actor.state_dict())==actor_hash,
                    "initial_critic_hash_unchanged":tensor_hash(init_critic.state_dict())==critic_hash}
            ck={"version":VERSION,"arm":name,"lambda":lam,"config":asdict(cfg),
                "actor":{k:v.detach().cpu().clone() for k,v in actor.state_dict().items()},
                "critic":{k:v.detach().cpu().clone() for k,v in critic.state_dict().items()},
                "actor_hash":tensor_hash(actor.state_dict()),
                "critic_hash":tensor_hash(critic.state_dict())}
            torch.save(ck,run/(name+'.pt'))
        save_json(run/'training.json',report)
        save_json(run/'math.json',gae_examples())
        save_json(run/'time_limit_bias.json',exact_time_limit_bias(gamma=cfg.gamma,horizon=cfg.rollout_horizon))
        save_json(run/'exact_policy_example.json',{
            "uniform_forward_probability":.5,"gamma":cfg.gamma,"horizon":cfg.rollout_horizon,
            "continuing_values":exact_policy_values([.5]*GOAL,cfg.gamma),
            "finite_horizon_success_probability":finite_horizon_success_probability([.5]*GOAL,cfg.rollout_horizon)})
        save_json(run/'reload.json',reload_run(run))
        manifest['status']='completed';save_json(run/'manifest.json',manifest)
        print('完成：',run.resolve(),flush=True)
        return run
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        save_json(run/'manifest.json',manifest)
        raise


def reload_run(path):
    run=Path(path)
    require(run.is_dir(),"缺少运行目录")
    report=json.loads((run/'training.json').read_text(encoding='utf-8'))
    outputs={}
    for name,entry in sorted(report['arms'].items()):
        ck=torch.load(run/(name+'.pt'),map_location='cpu',weights_only=True)
        require(ck['version']==VERSION and ck['arm']==name,"检查点版本或名字错误")
        config=Config(**ck['config']);config.validate()
        actor=Actor(config.actor_hidden).eval().requires_grad_(False)
        critic=Critic(config.critic_hidden).eval().requires_grad_(False)
        require(tensor_hash(ck['actor'])==ck['actor_hash'] and
                tensor_hash(ck['critic'])==ck['critic_hash'], "检查点内部身份不匹配")
        actor.load_state_dict(ck['actor'],strict=True)
        critic.load_state_dict(ck['critic'],strict=True)
        result=evaluate_policy(actor,critic,config.gamma,config.rollout_horizon,config.eval_episodes)
        before=entry['final']
        error=max(abs(result[key]-before[key]) for key in ('exact_success_within_limit','value_rmse'))
        require(error < 1e-9 and result['forward_probabilities']==before['forward_probabilities'],
                "重载结果不一致")
        outputs[name]={"max_abs_error":error,"parameters_trainable":sum(p.numel() for p in
                        list(actor.parameters())+list(critic.parameters()) if p.requires_grad)}
    return {"status":"passed","arms":outputs,"scope":"trusted_inference_reload_not_exact_resume"}

In [17]:
RUN_DIR=None
if RUN_FULL_TRAINING:
    RUN_DIR=experiment(Config(),OUTPUT_ROOT,mode='all')
    print('报告目录：',RUN_DIR)
else:
    print('明确跳过完整训练；不填入假模型结果。')

训练分支 td0 lambda= 0.0


训练分支 gae lambda= 0.95


训练分支 lambda1 lambda= 1.0


完成： /mnt/data/_lesson61_notebook_run/lesson_61_outputs/run_20261103T075157_435486Z


报告目录： lesson_61_outputs/run_20261103T075157_435486Z


## 11. 如何阅读实际结果？

本讲义附带的CPU默认实验中，初始六步到达概率约0.3675。GAE λ=0.95 与 λ=1 都学到了几乎总是前进的策略，但 **TD(0) 在这一个随机配置中几乎总是卡在中途**，其 Critic 也偏离精确价值。

这不是“TD(0) 理论上一定失败”的结论。它展示近似 Critic、稀疏延迟奖励与优化动力学可以耦合；单种子、固定超参数不允许做普遍算法排名。

下方只读取**本次真实运行**的报告，不写死测试成绩；不根据已观察的测试结果重新挑训练轮次。

In [18]:
if RUN_DIR is not None:
    report=json.loads((RUN_DIR/'training.json').read_text(encoding='utf-8'))
    init=report['reference_initial']
    print('起点：',round(init['exact_success_within_limit'],6),'值函数RMSE：',round(init['value_rmse'],6))
    print('分支          精确六步成功率    采样成功数   value RMSE')
    for name,row in sorted(report['arms'].items()):
        e=row['final']
        print(f"{name:12} {e['exact_success_within_limit']:.8f}   {e['sampled_successes']}/{e['sampled_episodes']}     {e['value_rmse']:.6f}")
else:
    print('未运行训练，没有报告可读取。')

起点： 0.367492 值函数RMSE： 0.974714
分支          精确六步成功率    采样成功数   value RMSE
gae          1.00000000   200/200     0.002283
lambda1      1.00000000   200/200     0.001955
td0          0.00001330   0/200     0.178436


## 12. 固定评价与精确价值不是一回事

价值 RMSE 比较的是 **当前 Actor 在底层继续任务上的预期折扣奖励**。六步成功概率则是另一个明确的外部限制指标。

我们没有把外部六步上限写进底层 $V^\pi$；如果你希望优化的是一个**真正仅有六步的有限时域问题**，就应把剩余时间加入状态，且将自然到期当成终止。这需要另一个任务定义与目标。

此外，GAE λ=1 且截断使用正确 bootstrap 时，对**这一段**有 $\widehat A+V=G^{bootstrap}$。它不是所有任务下的无限真实 Monte Carlo 回报，也不意味着不存在价值估计误差。

In [19]:
if RUN_DIR is not None:
    for name,row in sorted(report['arms'].items()):
        print(name)
        print('  前进概率：',[round(x,6) for x in row['final']['forward_probabilities']])
        print('  价值函数RMSE：',round(row['final']['value_rmse'],6))
        print('  最后一次记录的训练截断数：',row['history'][-1]['truncations_in_train'])

gae
  前进概率： [0.999561, 0.998645, 0.997299]
  价值函数RMSE： 0.002283
  最后一次记录的训练截断数： 0
lambda1
  前进概率： [0.998241, 0.998983, 0.997192]
  价值函数RMSE： 0.001955
  最后一次记录的训练截断数： 0
td0
  前进概率： [0.999995, 0.000715, 0.001865]
  价值函数RMSE： 0.178436
  最后一次记录的训练截断数： 24


## 13. 保存与独立重载

每次完整实验生成新的目录：

| 文件 | 内容 |
|---|---|
| `td0.pt` / `gae.pt` / `lambda1.pt` | 各自的 Actor/Critic 权重、配置、身份摘要 |
| `training.json` | 初始与三组最终评测、历史曲线、固定采样调度摘要 |
| `math.json`、`time_limit_bias.json` | 手算与概率精确枚举 |
| `exact_policy_example.json` | 固定策略的 Bellman 值与六步成功概率 |
| `reload.json` | 从干净网络加载权重后的结果对照 |
| `checks.json`、`manifest.json` | 机制检查及版本范围 |

模型文件用 `weights_only=True` 加载可信的本课检查点，**不是对未知来源文件的安全保证**。不含优化器与随机游标，不能作为精确断点续训状态。

命令行重载：

```bash
python lesson_61_value_functions_returns_and_gae.py --reload-run lesson_61_outputs/实际运行目录
```

In [20]:
if RUN_DIR is not None:
    print(json.dumps(reload_run(RUN_DIR),ensure_ascii=False,indent=2))

{
  "status": "passed",
  "arms": {
    "gae": {
      "max_abs_error": 0.0,
      "parameters_trainable": 0
    },
    "lambda1": {
      "max_abs_error": 0.0,
      "parameters_trainable": 0
    },
    "td0": {
      "max_abs_error": 0.0,
      "parameters_trainable": 0
    }
  },
  "scope": "trusted_inference_reload_not_exact_resume"
}


## 14. 自动检查：除了运行成功，还验证哪些错误必须拒绝？

固定检查覆盖：step 转移与奖励、真实终止与截断、回合连续性、Return 尾部、GAE 三个 λ 边界、Bellman 精确参考、价值回归、actor 更新、奖励来源与旧行为概率等。

测试不是完整环境/算法安全认证。梯度裁剪与随机有限样本存在数值和优化问题，跨设备或 PyTorch 版本也不保证逐位重现。

In [21]:
class Checks:
    def __init__(self):
        self.names=[]

    def ok(self,name,condition):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self,name,actual,expected,atol=1e-9,rtol=1e-9):
        a=torch.as_tensor(actual,dtype=torch.float64)
        b=torch.as_tensor(expected,dtype=torch.float64)
        self.ok(name,torch.allclose(a,b,atol=atol,rtol=rtol))

    def raises(self,name,fn):
        try:
            fn()
        except (LessonError, ValueError, TypeError):
            self.names.append(name)
        else:
            raise AssertionError(name+": 预期拒绝")


def check_gae_core(gae_function):
    gamma=.9
    # 两步，都非terminal；第二步是外部切断，仍需对真正next state bootstrap。
    ep=[Step(0,0,-.01,0,False,False,-.69),Step(0,0,-.01,0,False,True,-.69)]
    v=[.6,.8];nv=[.8,.5]
    d0=-.01+gamma*.8-.6  # .11
    d1=-.01+gamma*.5-.8  # -.36
    for lam in [0.,.5,1.]:
        got=gae_function([ep],v,nv,gamma,lam)
        expect=[d0+gamma*lam*d1,d1]
        if not torch.allclose(torch.tensor(got['advantage'],dtype=torch.float64),
                              torch.tensor(expect,dtype=torch.float64),atol=1e-10):
            raise AssertionError('GAE递推、lambda或截断bootstrap不正确')
        if not torch.allclose(torch.tensor(got['lambda_target'],dtype=torch.float64),
                              torch.tensor([expect[0]+v[0],expect[1]+v[1]],dtype=torch.float64),atol=1e-10):
            raise AssertionError('lambda-return没有与旧值正确组合')


def run_checks():
    torch.set_num_threads(1)
    seed_all(6111)
    c=Checks()
    cfg=Config()
    cfg.validate()
    c.ok('默认配置不启动真实模型服务',cfg.rollout_horizon==6 and cfg.episodes_per_update==24)
    c.raises('配置拒绝lambda超界',lambda:Config(gae_lambda=1.1).validate())
    c.raises('配置拒绝bool步数',lambda:Config(updates=True).validate())
    c.raises('配置拒绝负entropy',lambda:Config(entropy_coef=-.1).validate())
    c.raises('配置拒绝零value权重',lambda:Config(value_weight=0).validate())
    c.raises('配置拒绝非有限学习率',lambda:Config(actor_lr=float('nan')).validate())
    c.raises('配置拒绝过长horizon',lambda:Config(rollout_horizon=20).validate())
    c.ok('GOAL为3且非终止状态有3个',GOAL==3)
    c.ok('wait维持位置',transition(1,0,0,6)==(1,-.01,False,False))
    c.ok('advance推进位置',transition(1,1,0,6)==(2,-.01,False,False))
    c.ok('真终止获得奖励',transition(2,1,5,6)==(3,.99,True,False))
    c.ok('到外部上限但未到goal时截断',transition(1,0,5,6)==(1,-.01,False,True))
    c.raises('转移动作类型拒绝',lambda:transition(1,True,0,6))
    c.raises('转移时间超界拒绝',lambda:transition(1,0,6,6))
    s=Step(0,1,-.01,1,False,False,-.69)
    validate_step(s)
    c.raises('step不接受假奖励',lambda:validate_step(Step(0,1,1.,1,False,False,-.69)))
    c.raises('step不接受错误转移',lambda:validate_step(Step(0,1,-.01,0,False,False,-.69)))
    c.raises('step不接受错误终止标记',lambda:validate_step(Step(2,1,.99,3,False,False,-.69)))
    c.raises('step不接受双终止',lambda:validate_step(Step(2,1,.99,3,True,True,-.69)))
    c.raises('step不接受bool动作',lambda:validate_step(Step(0,True,-.01,1,False,False,-.69)))
    c.raises('step不接受NaN logp',lambda:validate_step(Step(0,1,-.01,1,False,False,float('nan'))))
    ep=[Step(0,1,-.01,1,False,False,-.69),Step(1,0,-.01,1,False,True,-.69)]
    c.raises('提前截断后不能接下一步',lambda:validate_episode(ep+[ep[-1]],3))
    c.raises('截断必须发生在horizon边界',lambda:validate_episode(ep,3))
    c.raises('回合起点必须是位置0',lambda:validate_episode(ep[1:],1))
    c.raises('缺少尾部标记',lambda:validate_episode([s],6))
    valid_long=[Step(0,0,-.01,0,False,False,-.69),Step(0,1,-.01,1,False,False,-.69),
                Step(1,1,-.01,2,False,False,-.69),Step(2,1,.99,3,True,False,-.69)]
    validate_episode(valid_long,6)
    c.close('两步无尾部Return',discounted_returns([1.,2.],.5),[2.,2.])
    c.close('两步带bootstrapReturn',discounted_returns([1.,2.],.5,3.),[2.75,3.5])
    c.close('单步gamma等于1',discounted_returns([1.],1.),[1.])
    c.raises('拒绝空return',lambda:discounted_returns([],.9))
    c.raises('拒绝负gamma',lambda:discounted_returns([1.],-.1))
    c.raises('拒绝NaN尾部',lambda:discounted_returns([1.],.9,float('nan')))
    c.raises('拒绝字符串奖励',lambda:discounted_returns(['1'],.9))
    m=gae_examples()
    c.close('截断应自举TD残差',m['correct_truncation_delta'],.05)
    c.close('截断误终止的残差',m['wrong_truncation_delta'],-.71)
    c.close('截断误终止少了0.76',m['lost_bootstrap'],.76)
    c.close('GAE第二步不跨回合',m['gae']['advantage'][1],.05)
    c.close('GAE第一步精确递推',m['gae']['advantage'][0],
            m['gae']['delta'][0]+m['gamma']*m['lambda']*m['gae']['delta'][1])
    c.close('lambda0等于delta',m['lambda_0']['advantage'],m['lambda_0']['delta'])
    c.close('lambda1为带bootstrapReturn',m['lambda_1']['lambda_target'],m['mc_with_bootstrap'])
    check_gae_core(generalized_advantage_estimation);c.names.append('GAE多lambda手算契约')
    c.raises('GAE拒绝错配值数组',lambda:generalized_advantage_estimation([ep],[.5],[.6],.9,.95))
    c.raises('GAE拒绝非有限值',lambda:generalized_advantage_estimation([ep],[.5,float('nan')],[.6,.7],.9,.95))
    c.raises('GAE拒绝超界lambda',lambda:generalized_advantage_estimation([ep],[.5,.5],[.6,.7],.9,1.1))
    c.raises('GAE拒绝内部回合乱连',lambda:generalized_advantage_estimation([[ep[-1],ep[-1]]],[.5,.5],[.6,.7],.9,.95))
    # 在真正 terminal 处不引用next价值，传任意大值也不能泄漏。
    terminal=[Step(2,1,.99,3,True,False,-.69)]
    terminal_a=generalized_advantage_estimation([terminal],[.8],[100.],.95,1.)
    c.close('真终止不bootstrap',terminal_a['lambda_target'],[.99])
    c.close('真终止adv',terminal_a['advantage'],[.19])
    two=[Step(0,1,-.01,1,False,False,-.69),Step(1,0,-.01,1,False,True,-.69)]
    tr=generalized_advantage_estimation([two],[.4,.5],[.5,.6],.95,1.)
    c.close('截断lambda1的回报可有尾部',tr['lambda_target'],discounted_returns([-.01,-.01],.95,.6))
    # 外部截断的结束标记不等于底层MDP终止标记。
    enum=exact_time_limit_bias()
    c.close('枚举归一的继续价值',enum['proper_bootstrap_expected_return'],enum['true_continuing_value_start'])
    c.ok('误把外部截断当终止造成负偏差',enum['lost_expected_value']>.2)
    c.close('固定行为策略lambda各值期望为零',
        [x['mean_advantage_at_start'] for x in enum['lambda_gradient_noise_example'].values()],torch.zeros(4),atol=1e-10)
    c.ok('本例的GAE方差随lambda增加',list(x['variance_at_start'] for x in enum['lambda_gradient_noise_example'].values())==
         sorted(x['variance_at_start'] for x in enum['lambda_gradient_noise_example'].values()))
    for probs in ([0.,0.,0.],[1.,1.,1.],[.5,.5,.5]):
        c.ok('Bellman固定点_'+str(probs),all(-1. < x <= 1.1 for x in exact_policy_values(probs)))
        c.close('有限时限成功概率_'+str(probs),
                finite_horizon_success_probability(probs,6),
                0. if probs[0]==0 else 1. if probs[0]==1 else 42/64)
    c.close('全等待无限时域价值',exact_policy_values([0.]*GOAL),[STEP_COST/(1-.95)]*GOAL)
    c.raises('无限时域gamma1不允许',lambda:exact_policy_values([.5]*GOAL,gamma=1))
    c.raises('枚举过长拒绝',lambda:exact_time_limit_bias(horizon=20))
    c.raises('无效策略概率拒绝',lambda:exact_policy_values([-1,.5,.5]))
    actor=Actor(16);critic=Critic(16)
    c.ok('actor与critic不同参数对象',not {id(p) for p in actor.parameters()}&{id(p) for p in critic.parameters()})
    c.ok('actor输出两个动作logits',tuple(actor(torch.tensor([0,1,2])).shape)==(3,2))
    c.ok('critic输出单值',tuple(critic(torch.tensor([0,1,2])).shape)==(3,))
    c.raises('actor拒绝终止状态输入',lambda:actor(torch.tensor([3])))
    c.raises('critic拒绝float状态',lambda:critic(torch.tensor([0.])))
    rand=random.Random(6101)
    episodes=collect_rollouts(actor,12,6,rand)
    c.ok('采样12个合法回合',len(episodes)==12)
    c.ok('采样每个回合都有终止/截断',all(ep[-1].terminated or ep[-1].truncated for ep in episodes))
    c.ok('采样回合不会越过horizon',all(1<=len(ep)<=6 for ep in episodes))
    c.ok('采样奖励从环境获得',all(all(st.reward in (-.01,.99) for st in ep) for ep in episodes))
    x=compute_targets(episodes,critic,.95,.95,True)
    c.ok('目标包含所有实际步骤',len(x['advantages'])==sum(len(ep) for ep in episodes))
    c.ok('目标返回不含PyTorch图',all(type(v) is float for v in x['lambda_returns']))
    c.ok('目标包含新旧观测',len(x['values_old'])==len(x['next_values_old']))
    c.ok('真终止值为零',all(nv==0 for st,nv in zip([s for ep in episodes for s in ep],x['next_values_old']) if st.terminated))
    forced=[Step(0,0,-.01,0,False,t==5,-.69) for t in range(6)]
    correct=compute_targets([forced],critic,.95,1.,True)
    wrong=compute_targets([forced],critic,.95,1.,False)
    final_v=value_prediction(critic)[0]
    c.close('截断值差gamma乘value',correct['deltas'][-1]-wrong['deltas'][-1],.95*final_v)
    c.ok('错误截断处理会改lambda1回报',
         abs(correct['lambda_returns'][0]-wrong['lambda_returns'][0])>1e-5)
    opta=torch.optim.Adam(actor.parameters(),lr=.035)
    optc=torch.optim.Adam(critic.parameters(),lr=.02)
    actor_old=tensor_hash(actor.state_dict());critic_old=tensor_hash(critic.state_dict())
    before_pred=value_prediction(critic)
    s_tensor=torch.tensor(x['states'],dtype=torch.long)
    act_tensor=torch.tensor(x['actions'],dtype=torch.long)
    with torch.no_grad():
        l=F.log_softmax(actor(s_tensor),-1)
        selected=l.gather(-1,act_tensor[:,None]).squeeze(-1)
        ent=-(l.exp()*l).sum(-1)
        expected_actor_loss=float(-(selected*torch.tensor(x['advantages'])).mean()-.01*ent.mean())
        expected_value_loss=float(.5*(critic(s_tensor)-torch.tensor(x['lambda_returns'])).square().mean())
    stats=optimize_actor_critic(actor,critic,x,opta,optc,Config(actor_hidden=16,critic_hidden=16))
    c.close('actor loss符合固定优势及熵',stats['actor_loss'],expected_actor_loss,atol=1e-7)
    c.close('critic loss只回归固定lambda target',stats['value_loss'],expected_value_loss,atol=1e-7)
    c.ok('actor参数实际改变',tensor_hash(actor.state_dict())!=actor_old)
    c.ok('critic参数实际改变',tensor_hash(critic.state_dict())!=critic_old)
    c.ok('critic数值预测实际改变',value_prediction(critic)!=before_pred)
    c.ok('损失/梯度范数有限',all(math.isfinite(stats[k]) for k in ('actor_loss','value_loss','actor_grad_norm','critic_grad_norm')))
    c.ok('训练计数正确',stats['training_steps']==len(x['states']))
    c.raises('旧概率不匹配要拒绝',lambda:optimize_actor_critic(actor,critic,x,opta,optc,Config(actor_hidden=16,critic_hidden=16)))
    c.raises('优化器对象错配要拒绝',lambda:optimize_actor_critic(Actor(16),critic,x,opta,optc,Config(actor_hidden=16,critic_hidden=16)))
    # 接口在本地固定种子下可复现，无需把训练表现写死为特定值。
    seed_all(6111)
    a1=Actor(16);v1=Critic(16)
    seed_all(6111)
    a2=Actor(16);v2=Critic(16)
    c.ok('同种子初始化逐位相同',tensor_hash(a1.state_dict())==tensor_hash(a2.state_dict()) and
         tensor_hash(v1.state_dict())==tensor_hash(v2.state_dict()))
    c.ok('行为概率正规化',all(0 < p < 1 for p in actor_probabilities(a1)))
    c.ok('精确有限成功率在0到1',0<=evaluate_policy(a1,v1,.95,6,20)['exact_success_within_limit']<=1)
    # 后面训练不会使用测试侧选择；这里只有固定小实验验证训练会改变参数。
    schedule=[100,101,102]
    a3,v3,h=train_one_arm(Config(actor_hidden=16,critic_hidden=16,updates=3,episodes_per_update=12),
                          .95,schedule,a1,v1)
    c.ok('简短训练记录首末回合',h[0]['update']==1 and h[-1]['update']==3)
    c.ok('训练器不更改起点网络',tensor_hash(a1.state_dict())==tensor_hash(a2.state_dict()))
    c.ok('训练器能更新实际actor',tensor_hash(a3.state_dict())!=tensor_hash(a1.state_dict()))
    c.ok('训练器能更新实际critic',tensor_hash(v3.state_dict())!=tensor_hash(v1.state_dict()))
    return {"passed":len(c.names),"names":c.names,
            "scope":"math_exact_enumeration_and_cpu_actor_critic_not_language_model_benchmark"}

In [22]:
CHECKS=run_checks()
print('固定检查通过：',CHECKS['passed'])
if RUN_DIR is not None:
    save_json(RUN_DIR/'checks.json',CHECKS)

固定检查通过： 91


## 15. 独立练习 A：TD target

实现：`td_targets_ex(rewards,next_values,terminated,gamma)`，返回同长度列表。

只在真正 `terminated=True` 时把下一价值设为0；外部截断**不在这个函数的终止掩码中**。检查输入长度、布尔类型和 γ 的范围。

注意：这里输出的是TD target，**不是TD residual**；若要 residual，还需减去当前旧价值 $V(s_t)$。

In [23]:
def td_targets_ex(rewards,next_values,terminated,gamma):
    raise NotImplementedError('独立实现TD target与真终止掩码')

RUN_EXERCISE_A=False

## 16. 独立练习 B：GAE 反向递推

实现：`gae_recursion_ex(deltas,boundaries,gamma,lam)`。

`boundaries[t]=True` 表示该条后不连接下一条优势（无论真实终止还是外部截断）。传入的 delta 已经按正确规则计算好，它与 bootstrap 的职责分开。

从后往前递推，检查 λ=0 与 λ=1；不要把两段不同 episode 的优势连在一起。

In [24]:
def gae_recursion_ex(deltas,boundaries,gamma,lam):
    raise NotImplementedError('独立实现固定边界的GAE反向递推')

RUN_EXERCISE_B=False

## 17. 独立练习 C：Critic 的固定 λ-return 回归

实现：`value_loss_ex(predictions,old_values,advantages)`。

返回 $0.5\operatorname{mean}\big(V_{new}-(V_{old}+A)\big)^2$。`predictions` 保留梯度，但 old_values 和 advantages 不能接受训练梯度。输入形状必须匹配。

不要改成直接拟合奖励0/1；这份标签结合了旧价值与多步信息。

In [25]:
def value_loss_ex(predictions,old_values,advantages):
    raise NotImplementedError('独立实现固定lambda-return的Critic回归')

RUN_EXERCISE_C=False

In [26]:
def exercise_check_a(fn):
    c=Checks()
    c.close('正确TD目标',fn([-.01,.99,-.01],[.8,100.,.4],[False,True,False],.95),
            [-.01+.95*.8,.99,-.01+.95*.4])
    c.raises('无效长度拒绝',lambda:fn([1],[1,2],[False],.9))
    c.raises('terminated不接受数字标志',lambda:fn([1],[1],[1],.9))
    return len(c.names)

def exercise_check_b(fn):
    c=Checks()
    for lam in [0.,.5,1.]:
        want=[.2+.9*lam*(-.1),-.1,.4]
        c.close('边界分离_'+str(lam),fn([.2,-.1,.4],[False,True,True],.9,lam),want)
    c.raises('非法lambda拒绝',lambda:fn([1],[True],.9,1.5))
    return len(c.names)

def exercise_check_c(fn):
    c=Checks()
    pred=torch.tensor([.3,.5],dtype=torch.float64,requires_grad=True)
    old=torch.tensor([.2,.4],dtype=torch.float64,requires_grad=True)
    adv=torch.tensor([.1,-.1],dtype=torch.float64,requires_grad=True)
    got=fn(pred,old,adv)
    target=old.detach()+adv.detach()
    expected=.5*(pred-target).square().mean()
    c.close('value回归目标',got,expected)
    gp,go,ga=torch.autograd.grad(got,(pred,old,adv),allow_unused=True)
    c.close('critic梯度',(gp).detach(),torch.autograd.grad(expected,pred)[0])
    c.ok('旧值及优势不接收梯度',go is None and ga is None)
    c.raises('形状不一致拒绝',lambda:fn(pred,old[:1],adv))
    return len(c.names)

In [27]:
if RUN_EXERCISE_A: print('练习A：',exercise_check_a(td_targets_ex),'项通过')
if RUN_EXERCISE_B: print('练习B：',exercise_check_b(gae_recursion_ex),'项通过')
if RUN_EXERCISE_C: print('练习C：',exercise_check_c(value_loss_ex),'项通过')
if not any((RUN_EXERCISE_A,RUN_EXERCISE_B,RUN_EXERCISE_C)):
    print('练习尚待独立完成；默认不开启，不会伪造通过。')

练习尚待独立完成；默认不开启，不会伪造通过。


## 18. 参考实现与检查

请先做完自己的练习，再展开下一个单元。参考实现采用独立名称，不覆盖你的函数。

In [28]:
def td_targets_ref(rewards,next_values,terminated,gamma):
    checked_numeric(rewards,'rewards');checked_numeric(next_values,'next_values')
    require(len(rewards)==len(next_values)==len(terminated),"长度不一致")
    require(all(type(x) is bool for x in terminated),"terminated必须是bool")
    require(type(gamma) in (float,int) and 0<gamma<=1,"gamma错误")
    return [r+gamma*(0. if done else nv) for r,nv,done in zip(rewards,next_values,terminated)]

def gae_recursion_ref(deltas,boundaries,gamma,lam):
    checked_numeric(deltas,'deltas')
    require(len(deltas)==len(boundaries) and all(type(b) is bool for b in boundaries),"边界数据非法")
    require(0<gamma<=1 and 0<=lam<=1,"gamma/lambda非法")
    result=[0.]*len(deltas);a=0.
    for t in reversed(range(len(deltas))):
        a=deltas[t]+gamma*lam*(0. if boundaries[t] else a)
        result[t]=a
    return result

def value_loss_ref(predictions,old_values,advantages):
    require(isinstance(predictions,torch.Tensor) and predictions.ndim==1 and predictions.requires_grad,
            "predictions必须是一维可导张量")
    old=torch.as_tensor(old_values,dtype=predictions.dtype,device=predictions.device).detach()
    advantage=torch.as_tensor(advantages,dtype=predictions.dtype,device=predictions.device).detach()
    require(predictions.shape==old.shape==advantage.shape and predictions.numel()>0,"形状不一致")
    return .5*(predictions-(old+advantage).detach()).square().mean()

In [29]:
print('参考A：',exercise_check_a(td_targets_ref))
print('参考B：',exercise_check_b(gae_recursion_ref))
print('参考C：',exercise_check_c(value_loss_ref))

参考A： 3
参考B： 4
参考C： 4


## 19. 命令行与验收

完整脚本内置与本 Notebook 相同的实现；不依赖本 Notebook 中的临时变量。

```bash
python lesson_61_value_functions_returns_and_gae.py --checks-only
python lesson_61_value_functions_returns_and_gae.py --math-only
python lesson_61_value_functions_returns_and_gae.py
python lesson_61_value_functions_returns_and_gae.py --mode gae --updates 100
```

答题验收：

- 为什么截断应保留 $V(s')$，却不能继续使用下一回合的优势？
- 为什么 λ=0、λ=1 都不是对所有 Critic/任务的万能选择？
- 为什么 Actor 用优势、Critic 回归固定返回目标，不能让旧值随参数反传？
- 为什么 Critic 对当前策略的价值RMSE与固定六步成功概率是不同指标？
- 为什么本课 TD(0) 的单种子表现差，不能据此断言 TD(0) 总会失败？

**下一课：第62课——PPO中的 Actor/Critic 联合目标、Value Clipping 与 GAE 工程集成。** 将价值目标连接回上一课概率比/Clip 的完整训练批次。

## 参考资料

1. [Schulman 等：Generalized Advantage Estimation](https://arxiv.org/abs/1506.02438)：GAE、λ、优势方差/偏差。
2. [Gymnasium：Handling Time Limits](https://gymnasium.farama.org/v1.1.1/tutorials/gymnasium_basics/handling_time_limits/)：terminated vs truncated 与 bootstrap 规则。
3. [Spinning Up：Vanilla Policy Gradient](https://spinningup.openai.com/en/latest/algorithms/vpg.html)：Actor、价值回归、baseline/优势。
4. [Spinning Up：PPO 实现](https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html)：对样本尾部自举与 GAE 的实现示例。
5. [PyTorch Autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html)：停止梯度与参数更新。

所有实验数据和网络在本地生成；没有来自真实语言模型的正确率、FLOPs、延迟、显存或 GPU 测量。